In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("Student_performance.csv")

In [ ]:
df_model = df.copy()

In [ ]:
df_model.columns = df_model.columns.str.strip()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = df_model[col].str.strip()

# Drop ID & Leakage
if 'id' in df_model.columns: df_model.drop(columns=['id'], inplace=True)
if 'exam_score' in df_model.columns: df_model.drop(columns=['exam_score'], inplace=True)

In [ ]:
#  Ordinal Encoding
ordinal_maps = {
    'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
    'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
    'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2}
}
for col, mapping in ordinal_maps.items():
    df_model[col] = df_model[col].map(mapping)

# Binary Encoding
df_model['internet_access'] = df_model['internet_access'].map({'yes': 1, 'no': 0})


le = LabelEncoder()
cols_to_encode = ['gender', 'course', 'study_method']

for col in cols_to_encode:
    df_model[col] = le.fit_transform(df_model[col])

In [ ]:
df_model.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score,student_performance
0,0,21,0,1,7.91,98.8,0,4.9,1,3,0,0,78.3,Excellent
1,1,18,2,6,4.95,94.8,1,4.7,0,4,1,1,46.7,Poor
2,2,20,0,1,4.68,92.6,1,5.8,0,0,2,1,99.0,Excellent
3,3,19,1,1,2.00,49.5,1,8.3,1,1,2,1,63.9,Good
4,4,23,1,5,7.65,86.9,1,9.6,2,4,2,0,100.0,Excellent


In [ ]:
# Creating  Features
df_model['study_efficiency'] = df_model['study_hours'] * (df_model['class_attendance'] / 100)
df_model['consistency'] = df_model['study_hours'] * df_model['sleep_hours']
df_model['wellness_index'] = df_model['sleep_quality'] * df_model['sleep_hours']
df_model['risk_factor'] = df_model['exam_difficulty'] / (df_model['study_hours'] + 0.1)

# Removing Dependency
cols_to_remove = ['study_hours', 'class_attendance', 'sleep_hours']
df_model.drop(columns=cols_to_remove, inplace=True)

In [ ]:
# Target ko alag kiya (Text format mein hi hai)
y = df_model['student_performance']
X = df_model.drop(columns=['student_performance'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Mean of Scaled Data (Approx 0): {np.mean(X_train_scaled):.4f}")
print(f"Std Dev of Scaled Data (Approx 1): {np.std(X_train_scaled):.4f}")

Mean of Scaled Data (Approx 0): 0.0000
Std Dev of Scaled Data (Approx 1): 1.0000


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

model = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=42)


model.fit(X_train_scaled, y_train)


y_pred = model.predict(X_test_scaled)

# 4. Accuracy Check
acc = accuracy_score(y_test, y_pred)

print("\n" + "="*50)
print(f"FINAL MODEL ACCURACY: {acc*100:.2f}%")
print("="*50)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


FINAL MODEL ACCURACY: 99.72%

Classification Report:
              precision    recall  f1-score   support

     Average       0.99      0.99      0.99     31662
   Excellent       1.00      1.00      1.00     31261
        Good       1.00      1.00      1.00     31509
        Poor       1.00      1.00      1.00     31568

    accuracy                           1.00    126000
   macro avg       1.00      1.00      1.00    126000
weighted avg       1.00      1.00      1.00    126000

